# Task 2: Multi-Modal Cardiac Image Fusion
Student: Sabeeh | Roll number: 23K-0002

## 1. Imports and configuration
Put a corresponding CT slice at `data/sample_ct.png` and MRI slice at `data/sample_mri.png`.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

TASK_DIR = Path.cwd()
if TASK_DIR.name != "Task_2_Cardiac_Fusion" and (TASK_DIR / "Task_2_Cardiac_Fusion").exists(): TASK_DIR /= "Task_2_Cardiac_Fusion"
DATA_DIR, OUTPUT_DIR = TASK_DIR / "data", TASK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CT_PATH, MRI_PATH = DATA_DIR / "sample_ct.png", DATA_DIR / "sample_mri.png"

## 2. Load corresponding slices and inspect
Only these two sample images are required. Inputs are converted to grayscale before equalization.

In [ ]:
for path in (CT_PATH, MRI_PATH):
    if not path.is_file(): raise FileNotFoundError(f"Missing {path}. Place one corresponding CT/MRI pair in Task_2_Cardiac_Fusion/data/ as sample_ct.png and sample_mri.png.")
ct = cv2.imread(str(CT_PATH), cv2.IMREAD_GRAYSCALE)
mri = cv2.imread(str(MRI_PATH), cv2.IMREAD_GRAYSCALE)
if ct is None or mri is None: raise ValueError("Could not read CT/MRI. Use valid PNG/JPG/JPEG files.")

if np.unique(ct).size < 16 or np.unique(mri).size < 16:
    raise ValueError("The supplied images look like simple placeholder graphics, not CT/MRI scans. Replace them with real cardiac slices before processing.")

if mri.shape != ct.shape:
    print(f"Resizing MRI from {mri.shape[::-1]} to CT dimensions {ct.shape[::-1]} for an illustrative overlay only.")
    mri = cv2.resize(mri, (ct.shape[1], ct.shape[0]), interpolation=cv2.INTER_LINEAR)


## 3. Independent histogram equalization

In [ ]:
ct_eq = cv2.equalizeHist(ct)
mri_eq = cv2.equalizeHist(mri)
cv2.imwrite(str(OUTPUT_DIR / "ct_enhanced.png"), ct_eq)
cv2.imwrite(str(OUTPUT_DIR / "mri_enhanced.png"), mri_eq)

## 4. Colored heatmaps and weighted fusion
CT has weight 0.70 to preserve sharp anatomical edges; MRI has weight 0.30 to contribute internal soft-tissue variation. The overlay is illustrative and does not perform spatial registration.

In [ ]:
ct_heat = cv2.applyColorMap(ct_eq, cv2.COLORMAP_JET)
mri_heat = cv2.applyColorMap(mri_eq, cv2.COLORMAP_JET)
CT_WEIGHT, MRI_WEIGHT = 0.70, 0.30
fused = cv2.addWeighted(ct_heat, CT_WEIGHT, mri_heat, MRI_WEIGHT, 0)
cv2.imwrite(str(OUTPUT_DIR / "ct_heatmap.png"), ct_heat)
cv2.imwrite(str(OUTPUT_DIR / "mri_heatmap.png"), mri_heat)
cv2.imwrite(str(OUTPUT_DIR / "fused.png"), fused)

## 5. Logarithmic and power-law transformations
Apply to normalized fused color channels; logarithmic compression then gamma (<1) expands darker values. Clipping and uint8 conversion keep values valid for OpenCV.

In [ ]:
fused01 = fused.astype(np.float32) / 255.0
log_fused = np.log1p(fused01) / np.log(2.0)
GAMMA = 0.75
final01 = np.power(np.clip(log_fused, 0, 1), GAMMA)
final_transformed = np.round(final01 * 255).clip(0,255).astype(np.uint8)
cv2.imwrite(str(OUTPUT_DIR / "final_transformed.png"), final_transformed)

## 6. Compare standalone CT, MRI, and fused result; save outputs

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,5))
axes[0].imshow(ct_eq, cmap="gray", vmin=0, vmax=255); axes[0].set_title("Enhanced CT"); axes[0].axis("off")
axes[1].imshow(mri_eq, cmap="gray", vmin=0, vmax=255); axes[1].set_title("Enhanced MRI"); axes[1].axis("off")
axes[2].imshow(cv2.cvtColor(final_transformed,cv2.COLOR_BGR2RGB)); axes[2].set_title("Fused + log + gamma"); axes[2].axis("off")
plt.tight_layout(); fig.savefig(OUTPUT_DIR / "comparison.png",dpi=150,bbox_inches="tight"); plt.show()
print("Saved outputs to", OUTPUT_DIR.resolve())
